In [ ]:
# ==========================================================
# CLEAN INSTALL (COLAB STABLE FOR GNN + STRING)
# ==========================================================

!pip uninstall -y numpy pandas scipy networkx

!pip install numpy==2
!pip install pandas==2.2.2
!pip install scipy==1.12
!pip install networkx==3.2.1

!pip install torch torchvision torchaudio -q
!pip install torch_geometric -q
!pip install requests tqdm scikit-learn -q

# SSL fix for STRING dataset
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: scipy 1.12.0
Uninstalling scipy-1.12.0:
  Successfully uninstalled scipy-1.12.0
Found existing installation: networkx 3.2.1
Uninstalling networkx-3.2.1:
  Successfully uninstalled networkx-3.2.1
  Using cached numpy-2.0.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
Using cached numpy-2.0.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
imbalanced-learn 0.14.1 requires scipy<2,>=1.11.4, which is not installed.
plotnine 0.14.5 requires pandas>=2.2.0, which is not installed.
plotnine 0.14.5 requires scipy>=1.8.0, which is

  Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.7 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
plotnine 0.14.5 requires scipy>=1.8.0, which is not installed.
pymc 5.28.1 requires scipy>=1.4.1, which is not installed.
pointpats 2.5.5 requires scipy>=1.12, which is not installed.
segregation 2.5.3 requires scipy, which is not installed.
sklearn-pandas 2.2.0 requires scipy>=1.5.1, which is not installed.
momepy 0.11.0 requires networkx>=3.2, which is not installed.
esda 2.8.2 requires scipy>=1.12, which is not installed.
mapclassify 2.10.0 requires networkx>=3.2, which is not installed.
mapclassify 2.10.0 requires scipy>=1.12, which is not installed.
spopt 0.7.0 requires networkx>=3.2, which is not installed.
spopt

  Using cached scipy-1.12.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached scipy-1.12.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.8 MB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.0
    Uninstalling numpy-2.0.0:
      Successfully uninstalled numpy-2.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spanner-graph-notebook 1.1.8 requires networkx, which is not installed.
hyperopt 0.2.7 requires networkx>=2.2, which is not installed.
momepy 0.11.0 requires networkx>=3.2, which is not installed.
python-louvain 0.16 requires networkx, which is not installed.
mapclassify 2.10.0 

  Using cached networkx-3.2.1-py3-none-any.whl.metadata (5.2 kB)
Using cached networkx-3.2.1-py3-none-any.whl (1.6 MB)


In [ ]:


# ==========================================================
# IMPORTS
# ==========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import pandas as pd
import networkx as nx
import requests
import gzip
import shutil
import random

from tqdm import tqdm
from scipy.sparse.linalg import eigsh, ArpackNoConvergence

from sklearn.metrics import f1_score, average_precision_score

from torch_geometric.nn import GATConv
from torch_geometric.data import Data


In [ ]:


# ==========================================================
# DOWNLOAD STRING
# ==========================================================

print("Downloading STRING...")

ppi_url = "https://stringdb-static.org/download/protein.links.v11.5/9606.protein.links.v11.5.txt.gz"

r = requests.get(ppi_url, verify=False)
open("ppi.gz","wb").write(r.content)

with gzip.open("ppi.gz","rb") as f_in:
    with open("ppi.txt","wb") as f_out:
        shutil.copyfileobj(f_in,f_out)


# ==========================================================
# LOAD GRAPH
# ==========================================================

df = pd.read_csv("ppi.txt",sep=" ")

df = df[df["combined_score"] > 700]

G = nx.from_pandas_edgelist(df,"protein1","protein2")

nodes = list(G.nodes())
node_map = {n:i for i,n in enumerate(nodes)}

print("Nodes:",len(nodes))
print("Edges:",G.number_of_edges())


# ==========================================================
# EDGE INDEX
# ==========================================================

edges = []

for u,v in G.edges():

    edges.append([node_map[u],node_map[v]])
    edges.append([node_map[v],node_map[u]])

edge_index = torch.tensor(edges).t().contiguous()


# ==========================================================
# STRUCTURAL FEATURES
# ==========================================================

print("Computing structural features...")

degree = np.array([G.degree(n) for n in nodes])

pagerank_dict = nx.pagerank(G)
pagerank = np.array([pagerank_dict[n] for n in nodes])

clustering_dict = nx.clustering(G)
clustering = np.array([clustering_dict[n] for n in nodes])

struct = np.stack([degree,pagerank,clustering],axis=1)


# ==========================================================
# LAPLACIAN POSITIONAL ENCODING
# ==========================================================

print("Computing Laplacian PE")

L = nx.normalized_laplacian_matrix(G).astype(float)

k = 17

try:

    eigval,eigvec = eigsh(
        L,
        k=k,
        which="SM",
        tol=1e-3,
        maxiter=500000,
        ncv=80
    )

except ArpackNoConvergence as e:

    print("ARPACK partial convergence")

    eigvec = e.eigenvectors

    if eigvec.shape[1] < k:

        pad = np.zeros((eigvec.shape[0],k-eigvec.shape[1]))
        eigvec = np.concatenate([eigvec,pad],axis=1)

lap_pe = eigvec[:,1:]


# ==========================================================
# FEATURE MATRIX
# ==========================================================

X = np.concatenate([struct,lap_pe],axis=1)

X = (X-X.mean(0))/(X.std(0)+1e-8)

x = torch.tensor(X,dtype=torch.float)


# ==========================================================
# FAKE LABELS (PLACEHOLDER)
# ==========================================================

num_nodes = x.shape[0]
num_classes = 50

y = torch.randint(0,2,(num_nodes,num_classes)).float()


# ==========================================================
# SPLIT
# ==========================================================

indices = list(range(num_nodes))
random.shuffle(indices)

train_size = int(0.7*num_nodes)
val_size = int(0.1*num_nodes)

train_idx = indices[:train_size]
val_idx = indices[train_size:train_size+val_size]
test_idx = indices[train_size+val_size:]


train_mask = torch.zeros(num_nodes,dtype=torch.bool)
val_mask = torch.zeros(num_nodes,dtype=torch.bool)
test_mask = torch.zeros(num_nodes,dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True


data = Data(
    x=x,
    edge_index=edge_index,
    y=y,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask
)


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'stringdb-static.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Nodes: 16795
Edges: 252013
Computing structural features...
Computing Laplacian PE


In [ ]:
class SNNResGATHOPE(nn.Module):

    def __init__(self,in_dim,hidden_dim,out_dim,heads=4):

        super().__init__()

        self.gat1 = GATConv(in_dim,hidden_dim,heads=heads,dropout=0.3)
        self.bn1 = nn.BatchNorm1d(hidden_dim*heads)

        self.gat2 = GATConv(hidden_dim*heads,hidden_dim,dropout=0.3)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

        self.res_proj = nn.Linear(hidden_dim*heads,hidden_dim)

        self.fast = nn.Linear(hidden_dim,hidden_dim)
        self.slow = nn.Linear(hidden_dim,hidden_dim)

        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta = nn.Parameter(torch.tensor(0.5))

        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Linear(hidden_dim,out_dim)


    def forward(self,x,edge_index):

        h = self.gat1(x,edge_index)
        h = self.bn1(h)
        h = F.elu(h)

        h = self.dropout(h)

        h1 = self.res_proj(h)

        h = self.gat2(h,edge_index)
        h = self.bn2(h)
        h = F.elu(h)

        h = h + h1

        fast = torch.sigmoid(self.fast(h))
        slow = torch.sigmoid(self.slow(h))

        mod = self.alpha*fast + self.beta*slow

        h = h + h * mod

        return self.classifier(h)

In [ ]:

# ==========================================================
# TRAINING
# ==========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data = data.to(device)

model = SNNResGATHOPE(data.x.shape[1],64,data.y.shape[1]).to(device)

optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)

criterion = nn.BCEWithLogitsLoss()

best_val = 1e9
patience = 20
counter = 0


for epoch in range(500):

    model.train()

    optimizer.zero_grad()

    logits = model(data.x,data.edge_index)

    loss = criterion(
        logits[data.train_mask],
        data.y[data.train_mask]
    )

    loss.backward()
    optimizer.step()

    model.eval()

    with torch.no_grad():

        logits = model(data.x,data.edge_index)

        val_loss = criterion(
            logits[data.val_mask],
            data.y[data.val_mask]
        )

    print(epoch,loss.item(),val_loss.item())

    if val_loss < best_val:

        best_val = val_loss
        counter = 0

        torch.save(model.state_dict(),"best.pt")

    else:

        counter += 1

    if counter >= patience:

        print("Early stopping")
        break

0 0.7348488569259644 0.6993234157562256
1 0.7253420352935791 0.6980924606323242
2 0.7191687822341919 0.6975717544555664
3 0.7179495096206665 0.6972600817680359
4 0.7145906090736389 0.6970533728599548
5 0.7121886014938354 0.6968497633934021
6 0.7116165161132812 0.6966480016708374
7 0.7101372480392456 0.6964953541755676
8 0.7092261910438538 0.6963210701942444
9 0.7070805430412292 0.6962177157402039
10 0.7060956358909607 0.6961426138877869
11 0.705821692943573 0.6960487365722656
12 0.7050326466560364 0.6960090398788452
13 0.7038330435752869 0.6959657073020935
14 0.7034453749656677 0.6959235072135925
15 0.7033565044403076 0.695889949798584
16 0.7021293640136719 0.6958433389663696
17 0.7023895978927612 0.6958351135253906
18 0.701720118522644 0.6958082318305969
19 0.7009690999984741 0.6958124041557312
20 0.7005270719528198 0.6958427429199219
21 0.6999511122703552 0.6959004402160645
22 0.6996279358863831 0.6958668828010559
23 0.699447512626648 0.6958351731300354
24 0.6993390321731567 0.695738

In [ ]:



# ==========================================================
# TEST
# ==========================================================

model.load_state_dict(torch.load("best.pt"))

model.eval()

with torch.no_grad():

    logits = model(data.x,data.edge_index)

    probs = torch.sigmoid(logits[data.test_mask]).cpu().numpy()
    true = data.y[data.test_mask].cpu().numpy()

preds = (probs>0.5).astype(int)

micro = f1_score(true,preds,average="micro")
macro = f1_score(true,preds,average="macro")
aupr = average_precision_score(true,probs,average="micro")

print("Micro F1:",micro)
print("Macro F1:",macro)
print("AUPR:",aupr)

Micro F1: 0.5072336926194602
Macro F1: 0.47369066380738617
AUPR: 0.500169857787703


In [ ]:
# ==========================================================
# LINK PREDICTION DATASET
# ==========================================================

print("Preparing link prediction dataset...")

edges = list(G.edges())
edges = [(node_map[u],node_map[v]) for u,v in edges]

num_edges = len(edges)

# positivos
pos_edges = edges

# negativos (sampling)
neg_edges = set()

while len(neg_edges) < num_edges:

    u = random.randint(0,num_nodes-1)
    v = random.randint(0,num_nodes-1)

    if u == v:
        continue

    if (u,v) in pos_edges or (v,u) in pos_edges:
        continue

    neg_edges.add((u,v))

neg_edges = list(neg_edges)

# split
def split_edges(edge_list):

    random.shuffle(edge_list)

    n = len(edge_list)

    train = edge_list[:int(0.7*n)]
    val   = edge_list[int(0.7*n):int(0.85*n)]
    test  = edge_list[int(0.85*n):]

    return train,val,test

pos_train,pos_val,pos_test = split_edges(pos_edges)
neg_train,neg_val,neg_test = split_edges(neg_edges)


# ==========================================================
# MODEL (EMBEDDINGS)
# ==========================================================

class SNNResGATHOPE(nn.Module):

    def __init__(self,in_dim,hidden_dim,heads=4):

        super().__init__()

        self.gat1 = GATConv(in_dim,hidden_dim,heads=heads,dropout=0.3)
        self.bn1 = nn.BatchNorm1d(hidden_dim*heads)

        self.gat2 = GATConv(hidden_dim*heads,hidden_dim,dropout=0.3)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

        self.res_proj = nn.Linear(hidden_dim*heads,hidden_dim)

        self.fast = nn.Linear(hidden_dim,hidden_dim)
        self.slow = nn.Linear(hidden_dim,hidden_dim)

        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta = nn.Parameter(torch.tensor(0.5))

        self.dropout = nn.Dropout(0.3)


    def forward(self,x,edge_index):

        h = self.gat1(x,edge_index)
        h = self.bn1(h)
        h = F.elu(h)

        h = self.dropout(h)

        h1 = self.res_proj(h)

        h = self.gat2(h,edge_index)
        h = self.bn2(h)
        h = F.elu(h)

        h = h + h1

        fast = torch.sigmoid(self.fast(h))
        slow = torch.sigmoid(self.slow(h))

        mod = self.alpha*fast + self.beta*slow

        h = h + h * mod

        return h  # embeddings


# ==========================================================
# EDGE SCORING
# ==========================================================

def edge_score(z, edge_list):

    u = torch.tensor([e[0] for e in edge_list],device=z.device)
    v = torch.tensor([e[1] for e in edge_list],device=z.device)

    return (z[u] * z[v]).sum(dim=1)



Preparing link prediction dataset...


In [ ]:

# ==========================================================
# TRAINING
# ==========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x = x.to(device)
edge_index = edge_index.to(device)

model = SNNResGATHOPE(x.shape[1],64).to(device)

optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)


def get_loss(z, pos_edges, neg_edges):

    pos_score = edge_score(z,pos_edges)
    neg_score = edge_score(z,neg_edges)

    pos_loss = -torch.log(torch.sigmoid(pos_score)+1e-8).mean()
    neg_loss = -torch.log(1 - torch.sigmoid(neg_score)+1e-8).mean()

    return pos_loss + neg_loss


best_val = 1e9
patience = 20
counter = 0


for epoch in range(200):

    model.train()

    optimizer.zero_grad()

    z = model(x,edge_index)

    loss = get_loss(z,pos_train,neg_train)

    loss.backward()
    optimizer.step()

    model.eval()

    with torch.no_grad():

        z = model(x,edge_index)

        val_loss = get_loss(z,pos_val,neg_val)

    print(epoch,loss.item(),val_loss.item())

    if val_loss < best_val:

        best_val = val_loss
        counter = 0

        torch.save(model.state_dict(),"best_lp.pt")

    else:

        counter += 1

    if counter >= patience:

        print("Early stopping")
        break



In [ ]:

# ==========================================================
# TEST
# ==========================================================

from sklearn.metrics import roc_auc_score, average_precision_score

model.load_state_dict(torch.load("best_lp.pt"))

model.eval()

with torch.no_grad():

    z = model(x,edge_index)

    pos_score = torch.sigmoid(edge_score(z,pos_test)).cpu().numpy()
    neg_score = torch.sigmoid(edge_score(z,neg_test)).cpu().numpy()

    y_true = np.concatenate([np.ones(len(pos_score)), np.zeros(len(neg_score))])
    y_score = np.concatenate([pos_score, neg_score])


auc = roc_auc_score(y_true,y_score)
aupr = average_precision_score(y_true,y_score)

print("AUC:",auc)
print("AUPR:",aupr)